### Import des librairies utiles pour ce projet

In [343]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import plotly.express as px
import statsmodels.api as sm

### Chargement du dataset selon les csv remis lors de la mission

In [344]:
df_lapage=pd.read_csv(r'..\..\projet9_Lapage\data\processed\df_lapage.csv',sep=';', encoding='utf-8')

### Demandes d'Annabelle
 
Nous souhaitons élaborer différents graphiques autour du chiffre d'affaires comme par exemples l’évolution dans le temps du :

- chiffre d’affaires avec la moyenne mobile (tu pourras choisir la période : jour, semaine, mois, etc.),
- chiffre d’affaires par catégorie,
- nombre de clients par mois,
- nombre de transactions,
- nombre de produits vendus,
- etc.

Il serait également intéressant de faire un zoom sur les références :
- les tops,
- les flops,
- la répartition par catégorie,
- etc.

Enfin, j’aimerais avoir quelques informations sur les profils de nos clients :
- répartition du chiffre d'affaires pour les clients BtoB,
- courbe de Lorenz,
- etc.

Après, toutes les informations et tous graphiques qui apporteraient de
l’information pertinente sont les bienvenus !

In [345]:
print('Les informations minimales par colonnes du dataframe sont les suivantes :')
print(df_lapage.min())

print('\n ----------------------------------------------------------')
print('Les informations maximales par colonnes du dataframe sont les suivantes :')
print(df_lapage.max())

print('\n ----------------------------------------------------------')
print('Le type de chaque colonne du dataframe est :')
print(df_lapage.dtypes)

Les informations minimales par colonnes du dataframe sont les suivantes :
id_prod              0_0
date          2021-03-01
session_id           s_1
client_id            c_1
price               0.62
categ                  0
sex                    f
birth               1929
dtype: object

 ----------------------------------------------------------
Les informations maximales par colonnes du dataframe sont les suivantes :
id_prod             2_99
date          2023-02-28
session_id       s_99999
client_id          c_999
price              300.0
categ                  2
sex                    m
birth               2004
dtype: object

 ----------------------------------------------------------
Le type de chaque colonne du dataframe est :
id_prod        object
date           object
session_id     object
client_id      object
price         float64
categ           int64
sex            object
birth           int64
dtype: object


In [346]:
# Conversion de la colonen date en datetime
df_lapage['date'] = pd.to_datetime(df_lapage['date'])

#### Chiffre d’affaires avec la moyenne mobile : jour, semaine, mois

In [347]:
# Agregation en jour pour ensuite effectuer la moyenne glissante en semaine 
df_ca = df_lapage.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().to_frame().reset_index()

In [348]:
# Création de la colonne indiquant une moyenne glissante sur une semaine
df_ca['price (semaine)']=df_ca['price'].rolling(window=7).mean().round(2)

In [349]:
# Création de la colonne indiquant une moyenne glissante sur un mois
df_ca['price (mois)']=df_ca['price'].rolling(window=30).mean().round(2)

In [350]:
# Vérification de réelle création des colonnes de moyennes glissantes
df_ca.head(35)

,date,price,price (semaine),price (mois)
0,2021-03-01,16565.22,NaN,NaN
1,2021-03-02,15486.45,NaN,NaN
2,2021-03-03,15198.69,NaN,NaN
3,2021-03-04,15196.07,NaN,NaN
4,2021-03-05,17471.37,NaN,NaN
5,2021-03-06,15785.28,NaN,NaN
6,2021-03-07,14760.20,15780.47,NaN
7,2021-03-08,15679.53,15653.94,NaN
8,2021-03-09,15710.51,15685.95,NaN
9,2021-03-10,15496.87,15728.55,NaN


#### Chiffre d’affaires par catégorie + évolution dans le temps

In [351]:
df_lapage_mois = df_lapage.groupby([pd.Grouper(key='date', freq='ME'), 'categ', 'id_prod', 'session_id', 'client_id', 'sex', 'birth' ])['price'].sum().reset_index()

In [352]:
# Vérification des catégories uniques présentes dans la colonne categ
df_lapage_mois.categ.unique()

array([0, 1, 2])

In [353]:
for categ in df_lapage_mois.categ.unique():
    ca = df_lapage_mois[df_lapage_mois['categ'] == categ]['price'].sum()
    print (f'Le chiffre d\'affaire total de la catégorie {categ} est de {ca:,.0f} euros'.replace(',',' '))

Le chiffre d'affaire total de la catégorie 0 est de 4 419 731 euros
Le chiffre d'affaire total de la catégorie 1 est de 4 827 657 euros
Le chiffre d'affaire total de la catégorie 2 est de 2 780 275 euros


In [354]:
df_lapage_mois.head()

,date,categ,id_prod,session_id,client_id,sex,birth,price
0,2021-03-31,0,0_0,s_10114,c_1609,m,1980,3.75
1,2021-03-31,0,0_0,s_10750,c_7253,f,1974,3.75
2,2021-03-31,0,0_0,s_10842,c_8285,f,1955,3.75
3,2021-03-31,0,0_0,s_11105,c_8154,f,1989,3.75
4,2021-03-31,0,0_0,s_12813,c_2699,f,1989,3.75


#### Nombre de clients par mois + évolution dans le temps

#### Nombre de transactions + évolution dans le temps

#### Nombre de produits vendus + évolution dans le temps

In [355]:
# tous les produits sur toute la période, puis par mois, puis par semaine